# Inflation-linked bonds

**Prerequisites:** **05**. **`inflation_linked_bond`** needs a **real discount** curve plus an **`inflation_indices`** series in the market JSON (Python `MarketContext` helpers for CPI may be limited—patch the snapshot).


## Concept

- **Real coupon** plus **index ratio** from CPI fixes cashflows.
- **Real yield** and **breakeven inflation** are interpreted relative to nominal curves (this notebook prints registry metrics when available).
- Inflation index JSON must use **`lag: "none"`** on the index object when the bond applies its own lag (avoids double-lagging).


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, MarketContext

print("Imports OK (inflation-linked).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

ilb = json.dumps(
    {
        "type": "inflation_linked_bond",
        "spec": {
            "id": "TIPS-DEMO",
            "notional": {"amount": "1000000", "currency": "USD"},
            "issue_date": "2024-01-15",
            "maturity": "2034-01-15",
            "real_coupon": "0.025",
            "frequency": {"count": 6, "unit": "months"},
            "day_count": "Thirty360",
            "bdc": "following",
            "calendar_id": "weekends_only",
            "stub": "None",
            "base_date": "2024-01-15",
            # Matches synthetic US-CPI linear series at issue (see MarketContext cell; ~277.07 at 2024-01-15).
            "base_index": 277.0676975464149,
            "indexation_method": "TIPS",
            "inflation_index_id": "US-CPI",
            "lag": {"months": 3},
            "deflation_protection": "MaturityOnly",
            "discount_curve_id": "USD-TIPS",
            "quoted_clean": 100.0,
            "attributes": {"tags": [], "meta": {}},
            "pricing_overrides": {},
        },
    }
)

try:
    validate_instrument_json(ilb)
    print("ILB JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
tips = DiscountCurve(
    "USD-TIPS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85), (10.0, 0.70)],
    day_count="act_365f",
)
mc = MarketContext().insert(tips)
state = json.loads(mc.to_json())
# Synthetic CPI path below: keep in sync with `base_index` on the ILB (issue 2024-01-15 ≈ 277.068).
obs = []
y, m, v = 2019, 10, 250.0
for _ in range(80):
    obs.append([f"{y:04d}-{m:02d}-01", v])
    m += 1
    if m > 12:
        m, y = 1, y + 1
    v *= 1.002
state["inflation_indices"] = [
    {
        "id": "US-CPI",
        "currency": "USD",
        "observations": obs,
        "interpolation": "linear",
        "lag": "none",
        "seasonality": None,
    }
]
market_json = json.dumps(state)
print("inflation index observations:", len(obs))


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(ilb, market_json, AS_OF_STR, model="discounting")
    print(ValuationResult.from_json(out))
except Exception as exc:
    print("price:", type(exc).__name__, exc)


## Metrics


In [ ]:
try:
    out = price_instrument_with_metrics(
        ilb,
        market_json,
        AS_OF_STR,
        model="discounting",
        metrics=["breakeven_inflation", "index_ratio"],
    )
    vr = ValuationResult.from_json(out)
    for m in ("breakeven_inflation", "index_ratio"):
        v = vr.get_metric(m)
        if v is not None:
            print(m, ":", v)
    print("inflation metrics pass complete")
except Exception as exc:
    print("metrics:", type(exc).__name__, exc)


## Takeaways

- **Inflation indices** ride alongside curves in the v2 market snapshot.
- **Breakeven** and **index ratio** metrics connect linker pricing to macro intuition.
- Match **lag policy** between bond spec and index payload to avoid double-lag validation errors.
